In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [3]:
data = pd.read_csv('../../data/ham_data.csv')

In [4]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611124...,6.111247e+12,Fromage Blanc Nature – Milky Food Professional...,1 kg,Plastic,Milky Food Professional,"Dairies, ,, Fermented foods, ,, Fermented milk...",Vegetarian,Maroc,Maroc,...,-2.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0
1,https://world.openfoodfacts.org/product/611103...,6.111035e+12,sidi ali – سيدي علي – 33 cl,33 cl,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,https://world.openfoodfacts.org/product/611103...,6.111035e+12,"Eau minérale naturelle – sidi ali – 1,5 L","1,5 L","Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2 L,NaN,Sidi Ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,https://world.openfoodfacts.org/product/327408...,3.274080e+12,Eau De Source – Cristaline – 1500 ml,1500 ml,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",Cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)
data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip().lower() if pd.notnull(x) else x)

In [6]:
data[data["yesil_skor_notu"].isna()].sample(10)

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan,urun_bilgisi,birim,kategori_listesi,etiketler_listesi
58492,https://world.openfoodfacts.org/product/500011...,5.000113e+12,Sabor Original – Coca-Cola – 330 ml,330.0,NaN,coca-cola,"Beverages and beverages preparations, ,, Bever...","Green Dot, ,, Made in Portugal, ,, Natural fla...",NaN,Portugal,...,9.0,0.0,0.0,0.0,0.0,0.0,Sabor Original,ml,"[Beverages and beverages preparations, Beverag...","[Green Dot, Made in Portugal, Natural flavors,..."
21834,https://world.openfoodfacts.org/product/071077...,7.107798e+11,Protein Shake – LEAN BODY,NaN,NaN,lean body,Beverages,"No gluten, ,, No lactose",NaN,NaN,...,0.0,0.0,0.0,7.0,0.0,0.0,Protein Shake,NaN,[Beverages],"[No gluten, No lactose]"
17267,https://world.openfoodfacts.org/product/200406...,2.004060e+12,Aràndanos – Aldi – 300 g,300.0,NaN,aldi,es:Frutos rojos,NaN,NaN,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,Aràndanos,g,[es:Frutos rojos],[]
4470,https://world.openfoodfacts.org/product/356007...,3.560071e+12,Penne rigate – Carrefour – 500 g,500.0,"Plastic, ,, Bag",carrefour,"Plant-based foods and beverages, ,, Plant-base...","Distributor labels, ,, Carrefour Quality, ,, G...",European Union and Non European Union,Espagne,...,1.0,0.0,0.0,5.0,0.0,0.0,Penne rigate,g,"[Plant-based foods and beverages, Plant-based ...","[Distributor labels, Carrefour Quality, Green ..."
43365,https://world.openfoodfacts.org/product/431150...,3.760196e+12,Pur Jus Orange Pomme Bib 3L Cœur de Pom – 3 l,3.0,NaN,coeur de pom',"Plant-based foods and beverages, ,, Beverages,...","No preservatives, ,, No added sugar, ,, No col...",France,France,...,8.0,0.0,0.0,0.0,0.0,6.0,Pur Jus Orange Pomme Bib 3L Cœur de Pom,l,"[Plant-based foods and beverages, Beverages, P...","[No preservatives, No added sugar, No coloring..."
16756,https://world.openfoodfacts.org/product/341294...,3.412940e+12,Crème de noix de cajou – Senfas – 300 g,300.0,NaN,senfas,"Plant-based foods and beverages, ,, Plant-base...","Organic, ,, Vegetarian, ,, Vegan, ,, The Vegan...",NaN,NaN,...,1.0,NaN,0.0,7.0,1.0,0.0,Crème de noix de cajou,g,"[Plant-based foods and beverages, Plant-based ...","[Organic, Vegetarian, Vegan, The Vegan Society]"
22225,https://world.openfoodfacts.org/product/356470...,3.564707e+12,Bowl chevre –,NaN,NaN,NaN,"Meals, ,, Prepared salads","Nutriscore, ,, Nutriscore Grade A",NaN,NaN,...,1.0,1.0,1.0,1.0,0.0,2.0,Bowl chevre,NaN,"[Meals, Prepared salads]","[Nutriscore, Nutriscore Grade A]"
4378,https://world.openfoodfacts.org/product/356007...,3.560071e+12,Nos melanges gourmands – Carrefour – 400 g (2 ...,400.0,fr:Boite Carton - Sachet,carrefour,"Plant-based foods and beverages, ,, Plant-base...","Distributor labels, ,, Carrefour Quality, ,, M...",NaN,Italie,...,0.0,NaN,0.0,6.0,5.0,0.0,Nos melanges gourmands,g,"[Plant-based foods and beverages, Plant-based ...","[Distributor labels, Carrefour Quality, Made i..."
43648,https://world.openfoodfacts.org/product/501002...,4.388861e+12,Klare Gemüsebrühe – ja! – 140g,140.0,Glass Bottle with Plastic Lid,ja!,"Plant-based foods and beverages, ,, Plant-base...","Vegetarian, ,, Vegan",NaN,Germany,...,0.0,0.0,5.0,0.0,0.0,0.0,Klare Gemüsebrühe,g,"[Plant-based foods and beverages, Plant-based ...","[Vegetarian, Vegan]"
9809,https://world.openfoodfacts.org/product/201868...,2.018687e+07,Pão de Forma Integral – Certossa – 600 g,600.0,"Plastic, ,, Bag",certossa,"Plant-based foods and beverages, ,, Plant-base...","Green Dot, ,, pt:Ecoponto-amarelo",NaN,Portugal,...,0.0,0.0,4.0,4.0,4.0,0.0,Pão de Forma Integral,g,"[Plant-based foods and beverages, Plant-based ...","[Green Dot, pt:Ecoponto-amarelo]"


In [7]:
data[["miktar","birim"]].head()

,miktar,birim
0,1.0,kg
1,33.0,cl
2,1.5,l
3,2.0,l
4,1500.0,ml


In [8]:
data.isnull().mean() * 100

url                              0.000000
barkod                           0.001434
urun_adi                         0.002868
miktar                          19.241029
ambalaj                         58.770043
markalar                         3.513754
kategoriler                      0.002868
etiketler                       30.545277
mensei                          76.027594
uretim_yerleri                  80.076299
satildigi_ulkeler                0.108998
icerik_metni                    12.884720
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                   13.198807
nutriscore_notu                  0.103261
nova_grubu                      15.433267
yesil_skor_notu                 27.038694
palmiye_yagi_icermez            19.154978
vejetaryen                      22.860913
vegan_durumu                    12.669592
yag_seviyesi                     2.331985
doymus_yag_seviyesi              3.355993
seker_seviyesi                   2

In [9]:
data.drop(columns=["sodyum_g","enerji_kj","yag_seviyesi","etiketler","doymus_yag_seviyesi","seker_seviyesi","tuz_seviyesi","url","urun_adi","kategoriler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni","urun_bilgisi","nutriscore_puan","ns_negatif_puan","ns_pozitif_puan","ns_enerji_puan","ns_seker_puan","ns_doymus_yag_puan","ns_tuz_puan","ns_protein_puan","ns_lif_puan","ns_meyve_sebze_baklagil_puan"], inplace=True)


In [10]:
data["etiketler_listesi"].head(20)

0                                          [Vegetarian]
1                                                    []
2           [ISO 22000, ISO 14001, ISO 45001, ISO 9001]
3                                           [Green Dot]
4                                              [Triman]
5                                                    []
6                                                    []
7                                                    []
8                                    [Green Dot, Maroc]
9                                                    []
10                                          [Green Dot]
11    [French milk, Made in France, Nutriscore, Nutr...
12    [Fair trade, Source of fibre, High fibres, Mad...
13    [Vegetarian, Fair trade, No gluten, Organic, C...
14    [ISO 22000, ISO 14001, ISO 45001, ISO 9001, Na...
15    [Sustainable, No preservatives, Source of fibr...
16    [No gluten, No preservatives, FSC, Green Dot, ...
17                                              

In [11]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted([str(x) for x in unique_values])
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

barkod (69726): ['1.020304050607081e+17', '1000029852900.0', '10001219.0', '10001295.0', '10001356.0', '10001400.0', '10001404.0', '10001691.0', '10001707.0', '10001875.0'] ...
miktar (1176): ['0.0', '0.03', '0.042', '0.045', '0.046', '0.05', '0.055', '0.06', '0.065', '0.07'] ...
markalar (13209): ['"tradition culinaire"', "'z bregov", '(sans marque)', '07x netto 03.25', '1 2 3 fruits', '1 attimo in forma', '1 l', '1 x auer 01.25', '1%', '1-2-3'] ...
alerjenler (1942): ["['Acesulfame-potassium']", "['Apple', 'Banana', 'Celery', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Nuts']", "['Apple', 'Banana', 'Gluten', 'Sulphur dioxide and sulphites']", "['Apple', 'Banana', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Kiwi', 'Orange', 'Peach']", "['Apple', 'Banana', 'Milk']", "['Apple', 'Banana', 'Orange', 'Peach']", "['Apple', 'Banana', 'Orange', 'Sulphur dioxide and sulphites']"] ...


eser_miktarlar (2547): ["['0756yc051253']", "['10502909']", "['2023-00986-16-9-fl-oz-mcct']", "['250318d2ac033']", "['32950435704-ae']", "['535kt4c-2221']", "['Agua']", "['Aipo-amendoa', 'Pode-conter-vestigios-de-mestanda']", "['Ajonjoli', 'Huevo', 'Leche', 'Nuez']", "['Alcohol']"] ...
icerik_sayisi (163): ['1.0', '10.0', '100.0', '101.0', '102.0', '103.0', '104.0', '105.0', '106.0', '107.0'] ...
nutriscore_notu (6): ['A', 'B', 'C', 'D', 'E', 'nan']
nova_grubu (5): ['1.0', '2.0', '3.0', '4.0', 'nan']
yesil_skor_notu (6): ['A', 'B', 'C', 'D', 'E', 'nan']


palmiye_yagi_icermez (3): ['False', 'True', 'nan']
vejetaryen (3): ['False', 'True', 'nan']
vegan_durumu (4): ['maybe', 'nan', 'no', 'yes']
enerji_kcal (918): ['0.0', '1.0', '10.0', '100.0', '1000.0', '101.0', '1010.0', '102.0', '1023.0', '103.0'] ...


yag_g (4003): ['-0.880969080882353', '0.0', '0.0001', '0.0002984', '0.00049', '0.0005', '0.001', '0.00116', '0.0021', '0.0025'] ...
doymus_yag_g (3146): ['-0.126229194852941', '0.0', '0.0001', '0.000152', '0.0003030303030303', '0.0005', '0.001', '0.00105', '0.0015', '0.0016'] ...
karbonhidrat_g (4786): ['0.0', '0.0001', '0.0005', '0.0006666666666666', '0.001', '0.0018', '0.005', '0.0076', '0.009', '0.0099999997764826'] ...
seker_g (3780): ['0.0', '0.0001', '0.0003125', '0.0005', '0.0006', '0.0006666666666666', '0.001', '0.00115', '0.0015', '0.002'] ...
lif_g (3489): ['0.0', '0.00019619140625', '0.0002', '0.00027', '0.0003627232142857', '0.000375', '0.0004', '0.00045', '0.000475', '0.0005'] ...


protein_g (3554): ['0.0', '0.0001', '0.0003', '0.0004', '0.0006666666666666', '0.001', '0.0016', '0.002', '0.003', '0.00375'] ...
tuz_g (5329): ['-0.179538557970063', '0.0', '0.0001', '0.0001041666666666', '0.000107', '0.00011', '0.00011972', '0.000125', '0.00013', '0.000145'] ...
alkol_yuzde (156): ['0.0', '0.0001056689265727', '0.0005', '0.0035', '0.01', '0.0113636363636364', '0.0439', '0.05', '0.06', '0.0909090909090909'] ...


meyve_sebze_baklagil_yuzde (4789): ['-0.0042521158854143', '-0.013417968749998', '-0.03125', '-0.247395833333336', '-0.45', '-1.21732954545455', '0.0', '0.0001068115234375', '0.00018310546875', '0.0001836706090898'] ...
birim (235): ['a', 'adet', 'and', 'apples', 'b', 'bagels', 'bags', 'baguette', 'balls', 'barre'] ...


kategori_listesi (26057): ['["Aliments d\'origine végétale", \'Aliments et boissons à base de végétaux\', \'Aliments à base de fruits et de légumes\', \'Aliments à base de plantes en conserve\', \'Conserves\', \'Céréales en conserve\', \'Céréales et dérivés\', \'Céréales et pommes de terre\', \'Légumes en conserve\', \'Légumes et dérivés\', \'Maïs doux\', \'Maïs doux en conserve\', \'Maïs en conserve\', \'Vegetables - Prepared/Processed (Shelf Stable)\']', '["Aliments d\'origine végétale", \'Aliments et boissons à base de végétaux\', \'Aliments à base de fruits et de légumes\', \'Aliments à base de plantes séchées\', \'Fruits et produits dérivés\', \'Fruits secs\', \'Noix de coco sèches\', \'Produits déshydratés\']', '["Aliments d\'origine végétale", \'Aliments et boissons à base de végétaux\', \'Beurres de cacahuètes\', \'Beurres de légumineuses\', \'Légumineuses et dérivés\', \'Produits à tartiner\', "Purées d\'oléagineux", \'Pâtes à tartiner végétales\']', '["Aliments d\'origine vég

In [12]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     42772
False    13598
Name: count, dtype: int64

In [13]:
data.isnull().mean() * 100

barkod                         0.001434
miktar                        19.241029
markalar                       3.513754
alerjenler                     0.000000
eser_miktarlar                 0.000000
icerik_sayisi                 13.198807
nutriscore_notu                0.103261
nova_grubu                    15.433267
yesil_skor_notu               27.038694
palmiye_yagi_icermez          19.154978
vejetaryen                    22.860913
vegan_durumu                  12.669592
enerji_kcal                    1.223360
yag_g                          1.236268
doymus_yag_g                   2.309038
karbonhidrat_g                 1.330924
seker_g                        1.699510
lif_g                         29.167025
protein_g                      1.270688
tuz_g                          1.085678
alkol_yuzde                   95.133809
meyve_sebze_baklagil_yuzde    67.912974
birim                         21.181482
kategori_listesi               0.000000
etiketler_listesi              0.000000


In [14]:
data.to_csv('../../data/data.csv', index=False)

In [15]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'alerjenler', 'eser_miktarlar',
       'icerik_sayisi', 'nutriscore_notu', 'nova_grubu', 'yesil_skor_notu',
       'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu', 'enerji_kcal',
       'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g', 'lif_g',
       'protein_g', 'tuz_g', 'alkol_yuzde', 'meyve_sebze_baklagil_yuzde',
       'birim', 'kategori_listesi', 'etiketler_listesi'],
      dtype='str')